# ML Pipeline Preparation
Follow the instructions below to help you create your ML pipeline.
### 1. Import libraries and load data from database.
- Import Python libraries
- Load dataset from database with [`read_sql_table`](https://pandas.pydata.org/pandas-docs/stable/generated/pandas.read_sql_table.html)
- Define feature and target variables X and Y

In [1]:
# import libraries
import nltk
nltk.set_proxy("proxy.muc:8080")
nltk.download("punkt_tab")
nltk.download("wordnet")
import re
import numpy as np
import pandas as pd
from sqlalchemy import create_engine
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer


[nltk_data] Error loading punkt_tab: <urlopen error [Errno 11001]
[nltk_data]     getaddrinfo failed>
[nltk_data] Error loading wordnet: <urlopen error [Errno 11001]
[nltk_data]     getaddrinfo failed>


In [2]:
!pip show nltk

Name: nltk
Version: 3.9.1
Summary: Natural Language Toolkit
Home-page: https://www.nltk.org/
Author: NLTK Team
Author-email: nltk.team@gmail.com
License: Apache License, Version 2.0
Location: C:\Users\q556621\AppData\Roaming\Python\Python311\site-packages
Requires: click, joblib, regex, tqdm
Required-by: 


In [3]:
# load data from database
engine = create_engine('sqlite:///c:/Users/q556621/OneDrive - BMW Group/Data Scientist/Exercises/Project 2/Upload GitHub/data/Disaster_Response.db')
df = pd.read_sql_table("df",con=engine)
X = df["message"]
y = df.iloc[:,5:]

### 2. Write a tokenization function to process your text data

In [4]:
def tokenize(text):
    url_regex = 'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\(\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+'
    detected_urls = re.findall(url_regex, text)
    for url in detected_urls:
        text = text.replace(url, "urlplaceholder")

    tokens = word_tokenize(text)
    lemmatizer = WordNetLemmatizer()

    clean_tokens = []
    for tok in tokens:
        clean_tok = lemmatizer.lemmatize(tok).lower().strip()
        clean_tokens.append(clean_tok)

    return clean_tokens

### 3. Build a machine learning pipeline
This machine pipeline should take in the `message` column as input and output classification results on the other 36 categories in the dataset. You may find the [MultiOutputClassifier](http://scikit-learn.org/stable/modules/generated/sklearn.multioutput.MultiOutputClassifier.html) helpful for predicting multiple target variables.

In [6]:
pipeline = Pipeline([
        ('features', FeatureUnion([

            ('text_pipeline', Pipeline([
                ('vect', CountVectorizer(tokenizer=tokenize)),
                ('tfidf', TfidfTransformer())
            ])),

        ])),

        ('clf', RandomForestClassifier(n_jobs=8))
    ])

### 4. Train pipeline
- Split data into train and test sets
- Train pipeline

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, y)
pipeline.fit(X_train, y_train)


### 5. Test your model
Report the f1 score, precision and recall for each output category of the dataset. You can do this by iterating through the columns and calling sklearn's `classification_report` on each.

In [8]:
y_pred_test = pipeline.predict(X_test)
y_pred_train = pipeline.predict(X_train)

In [9]:
for i in range(y_test.shape[1]):
    print("---------------------- " + y.columns[i] + " ----------------------\n")
    print(classification_report(y_test.iloc[:,i].values, y_pred_test[:,i]))

---------------------- aid_related ----------------------

              precision    recall  f1-score   support

           0       0.73      0.93      0.81      3855
           1       0.83      0.51      0.63      2699

    accuracy                           0.75      6554
   macro avg       0.78      0.72      0.72      6554
weighted avg       0.77      0.75      0.74      6554

---------------------- medical_help ----------------------

              precision    recall  f1-score   support

           0       0.92      1.00      0.96      6000
           1       0.50      0.00      0.01       554

    accuracy                           0.92      6554
   macro avg       0.71      0.50      0.48      6554
weighted avg       0.88      0.92      0.88      6554

---------------------- medical_products ----------------------

              precision    recall  f1-score   support

           0       0.95      1.00      0.97      6207
           1       0.70      0.02      0.04       347


C:\Users\q556621\AppData\Roaming\Python\Python311\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\q556621\AppData\Roaming\Python\Python311\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\q556621\AppData\Roaming\Python\Python311\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitaliz

              precision    recall  f1-score   support

           0       0.99      1.00      0.99      6473
           1       0.00      0.00      0.00        81

    accuracy                           0.99      6554
   macro avg       0.49      0.50      0.50      6554
weighted avg       0.98      0.99      0.98      6554

---------------------- refugees ----------------------

              precision    recall  f1-score   support

           0       0.96      1.00      0.98      6317
           1       0.00      0.00      0.00       237

    accuracy                           0.96      6554
   macro avg       0.48      0.50      0.49      6554
weighted avg       0.93      0.96      0.95      6554

---------------------- death ----------------------

              precision    recall  f1-score   support

           0       0.96      1.00      0.98      6262
           1       0.75      0.01      0.02       292

    accuracy                           0.96      6554
   macro avg       

### 6. Improve your model
Use grid search to find better parameters. 

In [12]:
parameters = {
    'features__text_pipeline__vect__ngram_range': ((1, 1), (1, 2)),
    'clf__n_estimators': [50, 100, 200],
    'clf__min_samples_split': [2, 3, 4]
}

model = GridSearchCV(pipeline, param_grid=parameters, verbose=True, cv=2)
model.fit(X_train, y_train)


Fitting 2 folds for each of 18 candidates, totalling 36 fits


C:\Users\q556621\AppData\Roaming\Python\Python311\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
C:\Users\q556621\AppData\Roaming\Python\Python311\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
C:\Users\q556621\AppData\Roaming\Python\Python311\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
C:\Users\q556621\AppData\Roaming\Python\Python311\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
C:\Users\q556621\AppData\Roaming\Python\Python311\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will 

GridSearchCV(cv=2,
             estimator=Pipeline(steps=[('features',
                                        FeatureUnion(transformer_list=[('text_pipeline',
                                                                        Pipeline(steps=[('vect',
                                                                                         CountVectorizer(tokenizer=<function tokenize at 0x000001B701078CC0>)),
                                                                                        ('tfidf',
                                                                                         TfidfTransformer())]))])),
                                       ('clf',
                                        RandomForestClassifier(n_jobs=8))]),
             param_grid={'clf__min_samples_split': [2, 3, 4],
                         'clf__n_estimators': [50, 100, 200],
                         'features__text_pipeline__vect__ngram_range': ((1, 1),
                                                                        (1,
                                                                         2))},
             verbose=True)

In [13]:
model.best_params_

{'clf__min_samples_split': 2,
 'clf__n_estimators': 200,
 'features__text_pipeline__vect__ngram_range': (1, 2)}

{'clf__min_samples_split': 2,
 'clf__n_estimators': 200,
 'features__text_pipeline__vect__ngram_range': (1, 2)}

In [14]:
y_pred_test = model.predict(X_test)

### 7. Test your model
Show the accuracy, precision, and recall of the tuned model.  

Since this project focuses on code quality, process, and  pipelines, there is no minimum performance metric needed to pass. However, make sure to fine tune your models for accuracy, precision and recall to make your project stand out - especially for your portfolio!

In [15]:
for i in range(y_test.shape[1]):
    print("---------------------- " + y.columns[i] + " ----------------------\n")
    print(classification_report(y_test.iloc[:,i].values, y_pred_test[:,i]))

---------------------- aid_related ----------------------

              precision    recall  f1-score   support

           0       0.69      0.95      0.80      3786
           1       0.86      0.43      0.57      2768

    accuracy                           0.73      6554
   macro avg       0.78      0.69      0.69      6554
weighted avg       0.76      0.73      0.70      6554

---------------------- medical_help ----------------------

              precision    recall  f1-score   support

           0       0.92      1.00      0.96      6050
           1       0.60      0.01      0.01       504

    accuracy                           0.92      6554
   macro avg       0.76      0.50      0.49      6554
weighted avg       0.90      0.92      0.89      6554

---------------------- medical_products ----------------------

              precision    recall  f1-score   support

           0       0.95      1.00      0.97      6236
           1       0.36      0.01      0.02       318


C:\Users\q556621\AppData\Roaming\Python\Python311\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\q556621\AppData\Roaming\Python\Python311\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\q556621\AppData\Roaming\Python\Python311\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitaliz

### 8. Try improving your model further. Here are a few ideas:
* try other machine learning algorithms
* add other features besides the TF-IDF

In [16]:

nltk.download('averaged_perceptron_tagger_eng')

class StartingVerbExtractor(BaseEstimator, TransformerMixin):

    def starting_verb(self, text):
        sentence_list = nltk.sent_tokenize(text)
        for sentence in sentence_list:
            pos_tags = nltk.pos_tag(tokenize(sentence))
            first_word, first_tag = pos_tags[0]
            if first_tag in ['VB', 'VBP'] or first_word == 'RT':
                return True
        return False

    def fit(self, x, y=None):
        return self

    def transform(self, X):
        X_tagged = pd.Series(X).apply(self.starting_verb)
        return pd.DataFrame(X_tagged)


[nltk_data] Error loading averaged_perceptron_tagger_eng: <urlopen
[nltk_data]     error [Errno 11001] getaddrinfo failed>


In [17]:
new_pipeline = Pipeline([
        ('features', FeatureUnion([

            ('text_pipeline', Pipeline([
                ('vect', CountVectorizer(tokenizer=tokenize)),
                ('tfidf', TfidfTransformer())
            ])),

            ('starting_verb', StartingVerbExtractor())
        ])),

        ('clf', RandomForestClassifier(n_jobs=8))
    ])

new_pipeline.fit(X_train, y_train)


C:\Users\q556621\AppData\Roaming\Python\Python311\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


Pipeline(steps=[('features',
                 FeatureUnion(transformer_list=[('text_pipeline',
                                                 Pipeline(steps=[('vect',
                                                                  CountVectorizer(tokenizer=<function tokenize at 0x000001B701078CC0>)),
                                                                 ('tfidf',
                                                                  TfidfTransformer())])),
                                                ('starting_verb',
                                                 StartingVerbExtractor())])),
                ('clf', RandomForestClassifier(n_jobs=8))])

In [18]:

parameters = {
    'features__text_pipeline__vect__ngram_range': ((1, 1), (1, 2)),
    'clf__n_estimators': [50, 100, 200],
    'clf__min_samples_split': [2, 3, 4]
}

model = GridSearchCV(new_pipeline, param_grid=parameters,verbose=True,cv=2)
model.fit(X_train, y_train)

Fitting 2 folds for each of 18 candidates, totalling 36 fits


C:\Users\q556621\AppData\Roaming\Python\Python311\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
C:\Users\q556621\AppData\Roaming\Python\Python311\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
C:\Users\q556621\AppData\Roaming\Python\Python311\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
C:\Users\q556621\AppData\Roaming\Python\Python311\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
C:\Users\q556621\AppData\Roaming\Python\Python311\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will 

GridSearchCV(cv=2,
             estimator=Pipeline(steps=[('features',
                                        FeatureUnion(transformer_list=[('text_pipeline',
                                                                        Pipeline(steps=[('vect',
                                                                                         CountVectorizer(tokenizer=<function tokenize at 0x000001B701078CC0>)),
                                                                                        ('tfidf',
                                                                                         TfidfTransformer())])),
                                                                       ('starting_verb',
                                                                        StartingVerbExtractor())])),
                                       ('clf',
                                        RandomForestClassifier(n_jobs=8))]),
             param_grid={'clf__min_samples_split': [2, 3, 4],
                         'clf__n_estimators': [50, 100, 200],
                         'features__text_pipeline__vect__ngram_range': ((1, 1),
                                                                        (1,
                                                                         2))},
             verbose=True)

In [19]:
model.best_params_

{'clf__min_samples_split': 2,
 'clf__n_estimators': 200,
 'features__text_pipeline__vect__ngram_range': (1, 2)}

In [20]:
for i in range(y_test.shape[1]):
    print("---------------------- " + y.columns[i] + " ----------------------\n")
    print(classification_report(y_test.iloc[:,i].values, y_pred_test[:,i]))

---------------------- aid_related ----------------------

              precision    recall  f1-score   support

           0       0.69      0.95      0.80      3786
           1       0.86      0.43      0.57      2768

    accuracy                           0.73      6554
   macro avg       0.78      0.69      0.69      6554
weighted avg       0.76      0.73      0.70      6554

---------------------- medical_help ----------------------

              precision    recall  f1-score   support

           0       0.92      1.00      0.96      6050
           1       0.60      0.01      0.01       504

    accuracy                           0.92      6554
   macro avg       0.76      0.50      0.49      6554
weighted avg       0.90      0.92      0.89      6554

---------------------- medical_products ----------------------

              precision    recall  f1-score   support

           0       0.95      1.00      0.97      6236
           1       0.36      0.01      0.02       318


C:\Users\q556621\AppData\Roaming\Python\Python311\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\q556621\AppData\Roaming\Python\Python311\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\q556621\AppData\Roaming\Python\Python311\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitaliz

### 9. Export your model as a pickle file

In [21]:
import pickle

# save
with open("C:/Users/q556621/OneDrive - BMW Group/Data Scientist/Exercises/Project 2/Upload GitHub/models/model.pkl",'wb') as f:
    pickle.dump(model,f)

### 10. Use this notebook to complete `train_classifier.py`
Use the template file attached in the Resources folder to write a script that runs the steps above to create a database and export a model based on a new dataset specified by the user.